# **HMM2 · MOST LIKELY SEQUENCE OF STATES**

Introduce the same functions used fo HMM0 and HMM1, useful:

In [4]:
#same as hmm0
def read_matrix(line):
    # The first two numbers give the number of rows and columns.
    tokens = line.split()
    rows = int(tokens[0])
    cols = int(tokens[1])
    values = [float(token) for token in tokens[2:]]

    matrix = []
    for r in range(rows):
        start = r * cols
        end = start + cols
        matrix.append(values[start:end])

    return matrix

#same as hmm0
def vector_times_matrix(vector, matrix):
    result = []
    for j in range(len(matrix[0])):
        total = 0.0
        for i in range(len(vector)):
            total += vector[i] * matrix[i][j]
        result.append(total)
    return result

#same as hmm1
def elementwise_product(vector1, vector2):
    result = []
    for i in range(len(vector1)):
        result.append(vector1[i] * vector2[i])
    return result


Now define the algorithm:

In [5]:
#Viterbi algorithm
def viterbi(A, B, pi, observations):
    number_of_states = len(pi)

    # Step 1: Initialization. delta_1(i) = pi[i] * B[i][O_1]
    first_observation = observations[0]
    emission_column = [row[first_observation] for row in B]
    delta = elementwise_product(pi, emission_column)

    # psi[t][i] stores the best previous state for state i at time t. At Python index t = 0, there is no previous state.
    psi = [[None] * number_of_states]

    # Step 2: Recursion
    for observation in observations[1:]:

        best_values = []
        best_predecessors = []

        for i in range(number_of_states): # i is the current state (start with state 0 as a candidate). Iterate on all the states, 
            #find the one with greates product and save its value (best_value) and state pointer (best_state).
            best_state = 0
            best_value = delta[0] * A[0][i] 

            # Compare the other possible previous states j. -> [max (..,..,)]
            for j in range(1, number_of_states):
                candidate = delta[j] * A[j][i]
                if candidate > best_value:
                    best_value = candidate
                    best_state = j

            best_values.append(best_value)        # maximum
            best_predecessors.append(best_state)  # argmax

        # Multiply each maximum by the emission probability for state i 
        emission_column = [row[observation] for row in B]
        delta = elementwise_product(best_values, emission_column) #now get delta, wich is the max value of probability
        psi.append(best_predecessors) #and get psi, given by best predecessors

        # Keep values in a safe numerical range for long sequences.Dividing by the same number preserves the best path.
        scale = max(delta)
        if scale > 0:
            delta = [value / scale for value in delta]

    # Step 3: Select the best final state.
    last_state = delta.index(max(delta))

    # Step 4: Recover the path from the last state to the first.
    T = len(observations)
    path = [None] * T

    path[T - 1] = last_state # We already know the last state.

    for t in range(T - 1, 0, -1): # Fill in one previous state at a time.
        current_state = path[t]
        previous_state = psi[t][current_state]
        path[t - 1] = previous_state

    return path

And call it, by reading Kattis matrices:

In [6]:
def hmm2(input_text):
    lines = input_text.strip().splitlines()

    # Read the model exactly as in HMM1.
    A = read_matrix(lines[0])
    B = read_matrix(lines[1])
    pi = read_matrix(lines[2])[0]

    # The first number is the sequence length.
    tokens = lines[3].split()
    observations = [int(token) for token in tokens[1:]]

    path = viterbi(A, B, pi, observations)

    # Kattis expects the state indices on one line (do not output the length of the sequence).
    return " ".join(str(state) for state in path)

### Q4. The forward and Viterbi recurrences have almost the same structure, but one uses a sum and the other a maximum. Explain how this changes both the meaning of the stored values and the final output. Why are the backpointers ψt(i) needed?

Forward sums over all paths that end in state $i$, so $\alpha_t(i)$ includes their total probability. Summing the final values gives the probability of the observation sequence. 

Viterbi takes the maximum, so $\delta_t(i)$ stores the score of the best path ending in $i$. To recover that path, I choose the state with the highest final score and follow the backpointers backwards. Each $\psi_t(i)$ tells me which previous state gave the best score so It's really important to reconstruct the path.